## Load data

In [2]:
import pandas as pd
import numpy as np

#Load clean transaction data and targets (clean data)
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date","pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date","pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:",df_train.shape)
print("Valid transactions:",df_valid.shape)

Train transactions: (219128, 27)
Valid transactions: (55183, 27)


In [3]:
#Reference date: last day of the transaction period
#We use this to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")
reference_date

Timestamp('2017-12-31 00:00:00')

## RFM + pruschase behavior features

In [53]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input: transaction-level dataframe (one row per time)
    Output: one row per customer
    """
    df=df.copy()
    # Separate returned and non-returned items
    df_net = df[df["is_returned"]==0]
    
    #---- Recency: days since last purchase ----
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date -x).days)
        .rename("recency_days")
    )
    #---- Frequency: number of unique orders with at least one non-returned item ---
    #sale_id is the order identifier; one order can have multiple items
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )
    
    # --- Is the customer a one-time buyer? ---
    # Useful because one-time buyers are less likely to return
    is_one_time_buyer = (frequency == 1).astype(int).rename("is_one_time_buyer")
    
    # --- Average and std days between orders ---
    # Only meaningful for customers with more than one order
    # Customers with a single order will get NaN -- we'll fill later with median
    order_dates_per_cust = (
        df.drop_duplicates(subset=["cust_id", "sale_id"])  # one row per order
        .sort_values(["cust_id", "order_date"])
        .groupby("cust_id")["order_date"]
    )
    
    avg_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.mean() if len(x) > 1 else np.nan)
        .rename("avg_days_between_orders")
    )
    
    std_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.std() if len(x) > 1 else np.nan)
        .rename("std_days_between_orders")
    )

    #---- Monetary: total revenue (exluding returned items)---
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )
    
    #---- Average ticket: revenue per effective order  ---
    avg_ticket = (total_revenue/frequency).rename("avg_ticket")

    #---- Items per order (gross): includes returned items, captures order behavior ---
    items_per_order_gross = (
        df.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("irems_per_order_gross")
    )

    #--- Items per order (net): excludes returned items, captures what they actually kept---
    items_per_order_net = (
        df_net.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    
    #--- Return rate: absolute count and proportion proportion of items returned---
    total_returns = (
        df.groupby("cust_id")["is_returned"]
        .sum()
        .rename("total_returns")
    )

    total_items= (
        df.groupby("cust_id")["is_returned"]
        .count()
        .rename("total_items")
    )
    return_rate = (total_returns/total_items).rename("return_rate")

    #--- Average delivery time: days between order and packing ---
    df["delivery_days"]=(df["pack_date"]-df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    #--- Discount behaivor ---
    df["has_discount_item"] = (df["sale_discount_applied"]<0).astype(int)

    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"]
        .sum()
        .rename("total_discounted_items")
    )

    discount_rate= (
        df.groupby("cust_id")["has_discount_item"]
        .mean()
        .rename("discount_rate")
    )

    # Combine all features into one dataframe
    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket, items_per_order_gross, 
        items_per_order_net, total_returns, total_items, return_rate, 
        total_discounted_items, discount_rate, avg_days_between_orders, 
        std_days_between_orders, is_one_time_buyer ], axis=1).reset_index()
    return rfm

#Test it
rfm_train =compute_rfm_features(df_train)
print(rfm_train.shape)
rfm_train.head()
    
    

(93272, 15)


,cust_id,recency_days,frequency,total_revenue,avg_ticket,irems_per_order_gross,items_per_order_net,total_returns,total_items,return_rate,total_discounted_items,discount_rate,avg_days_between_orders,std_days_between_orders,is_one_time_buyer
0,222agnowc53dykbq,383,1.0,89.95,89.950000,1.000,1.000000,0,1,0.000000,0,0.000000,NaN,NaN,1.0
1,222ny4m63rmalpdl,374,1.0,125.93,125.930000,3.000,2.000000,1,3,0.333333,3,1.000000,NaN,NaN,1.0
2,223xvc4rbjatlnev,520,1.0,116.14,116.140000,2.000,2.000000,0,2,0.000000,2,1.000000,NaN,NaN,1.0
3,223y2r357elerbis,348,1.0,47.97,47.970000,2.000,1.000000,1,2,0.500000,2,1.000000,NaN,NaN,1.0
4,2245y4r7mgo45qg3,169,7.0,536.70,76.671429,1.125,1.142857,1,9,0.111111,8,0.888889,72.428571,51.529465,0.0


## Seasonality features

In [47]:
def compute_seasonality_features(df):
    """
    Computes seasonality features, aggregated at customer level.
    Captures when during the year the customer tends to buy
    Input: transaction-level dataframe
    Output: one row per customer
    """
    df = df.copy()

    #Extract month from order date
    df["order_month"]= df["order_date"].dt.month

    #--- Does the customer buy in January or July? (peak sales months)---
    buys_in_jan = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x: int((x==1).any()))
    .rename("buys_in_jan")
    )

    buys_in_jul = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x:int((x==7).any()))
    .rename("buys_in_jul")
    )

    #--- Most active month: the month where the customer placed most orders---
    #not sure about this cuz probably there will be a lot of ties
    most_active_month = (
    df.groupby(["cust_id", "order_month"])["sale_id"]
    .nunique()
    .groupby("cust_id").idxmax()
    .apply(lambda x:x[1])
    .rename("most_active_month")
    )

    #Combine
    seasonality = pd.concat([buys_in_jan, buys_in_jul, most_active_month], axis=1).reset_index()
    return seasonality

#Test it
seasonality_train = compute_seasonality_features(df_train)
print(seasonality_train.shape)
seasonality_train.head()

(93272, 4)


,cust_id,buys_in_jan,buys_in_jul,most_active_month
0,222agnowc53dykbq,0,0,12
1,222ny4m63rmalpdl,0,0,12
2,223xvc4rbjatlnev,0,1,7
3,223y2r357elerbis,1,0,1
4,2245y4r7mgo45qg3,1,1,2


## Product profile

In [48]:
def compute_product_features(df):
    """
    Computes product profile features, aggregated at customer level.
    Captures diversity of purchasing behavior across product categories,
    brands, and customer segments.
    Input: transaction-level dataframe (one row per item)
    Output: one row per customer
    """
    df = df.copy()

    # --- Number of distinct genders purchased for ---
    # A high number suggests a parent buying for the whole family
    n_genders = (
        df.groupby("cust_id")["prod_type_1"]
        .nunique()
        .rename("n_genders")
    )

    # --- Binary flags per gender ---
    buys_women = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "women").any()))
        .rename("buys_women")
    )

    buys_men = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "men").any()))
        .rename("buys_men")
    )

    buys_kids = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int(x.isin(["boys", "girls"]).any()))
        .rename("buys_kids")
    )

    # --- Diversity of product categories (prod_type_3: sneakers, boots, etc.) ---
    n_product_categories = (
        df.groupby("cust_id")["prod_type_3"]
        .nunique()
        .rename("n_product_categories")
    )

    # --- Number of distinct brands purchased ---
    n_brands = (
        df.groupby("cust_id")["prod_brand"]
        .nunique()
        .rename("n_brands")
    )

    # --- Proportion of web-only products purchased ---
    pct_web_only = (
        df.groupby("cust_id")["prod_web_only"]
        .mean()
        .rename("pct_web_only")
    )

    # --- Has the customer ever bought outlet products? ---
    buys_outlet = (
        df.groupby("cust_id")["prod_outlet"]
        .apply(lambda x: int((x > 0).any()))
        .rename("buys_outlet")
    )

    product_features = pd.concat([
        n_genders, buys_women, buys_men, buys_kids,
        n_product_categories, n_brands,
        pct_web_only, buys_outlet
    ], axis=1).reset_index()

    return product_features

# Test it
product_train = compute_product_features(df_train)
print(product_train.shape)
product_train.head()

(93272, 9)


,cust_id,n_genders,buys_women,buys_men,buys_kids,n_product_categories,n_brands,pct_web_only,buys_outlet
0,222agnowc53dykbq,1,0,1,0,1,1,0.000000,0
1,222ny4m63rmalpdl,2,0,1,1,2,2,0.000000,0
2,223xvc4rbjatlnev,1,1,0,0,1,1,0.000000,0
3,223y2r357elerbis,1,1,0,0,1,1,0.000000,0
4,2245y4r7mgo45qg3,4,1,1,1,5,7,0.222222,0


## Union

In [54]:
def build_features(df):
    """
    Builds the final customer-level feature table by combining
    all feature blocks: RFM, seasonality, and product profile.
    Input: transaction-level dataframe (one row per item)
    Output: one row per customer with all features
    """
    rfm = compute_rfm_features(df)
    seasonality = compute_seasonality_features(df)
    product = compute_product_features(df)

    # Merge all blocks on cust_id
    features = rfm.merge(seasonality, on="cust_id").merge(product, on="cust_id")

    return features

# Build features for train and valid
df_train_features = build_features(df_train)
df_valid_features = build_features(df_valid)

print("Train features shape:", df_train_features.shape)
print("Valid features shape:", df_valid_features.shape)
df_train_features.head()

Train features shape: (93272, 26)
Valid features shape: (23319, 26)


,cust_id,recency_days,frequency,total_revenue,avg_ticket,irems_per_order_gross,items_per_order_net,total_returns,total_items,return_rate,...,buys_in_jul,most_active_month,n_genders,buys_women,buys_men,buys_kids,n_product_categories,n_brands,pct_web_only,buys_outlet
0,222agnowc53dykbq,383,1.0,89.95,89.950000,1.000,1.000000,0,1,0.000000,...,0,12,1,0,1,0,1,1,0.000000,0
1,222ny4m63rmalpdl,374,1.0,125.93,125.930000,3.000,2.000000,1,3,0.333333,...,0,12,2,0,1,1,2,2,0.000000,0
2,223xvc4rbjatlnev,520,1.0,116.14,116.140000,2.000,2.000000,0,2,0.000000,...,1,7,1,1,0,0,1,1,0.000000,0
3,223y2r357elerbis,348,1.0,47.97,47.970000,2.000,1.000000,1,2,0.500000,...,0,1,1,1,0,0,1,1,0.000000,0
4,2245y4r7mgo45qg3,169,7.0,536.70,76.671429,1.125,1.142857,1,9,0.111111,...,1,2,4,1,1,1,5,7,0.222222,0


## Merge with target

In [55]:
# Merge features with targets
df_train_final = df_train_features.merge(df_train_targets, on="cust_id")
df_valid_final = df_valid_features.merge(df_valid_targets, on="cust_id")

print("Train final shape:", df_train_final.shape)
print("Valid final shape:", df_valid_final.shape)

Train final shape: (93272, 27)
Valid final shape: (23319, 27)


## Quick check

In [56]:
#Check target distribution
target = df_train_final["revenue_2018_2019"]
print("==== Target distribution ====")
print(f"Customers with revenue =0: {(target==0).sum()} ({100*(target==0).mean():.1f}%)")
print(f"Customers with revenue >0: {(target>0).sum()} ({100*(target>0).mean():.1f}%)")
print(f"\nMean (all): {target.mean():.2f}")
print(f"Mean (>0 only): {target[target>0].mean():.2f}")
print(f"Max: {target.max():.2f}")

#Check NaNs
print("\n=== NaNs per column ===")
nans= df_train_final.isna().sum()
print(nans[nans > 0])

==== Target distribution ====
Customers with revenue =0: 59196 (63.5%)
Customers with revenue >0: 34076 (36.5%)

Mean (all): 70.18
Mean (>0 only): 192.08
Max: 1197.94

=== NaNs per column ===
frequency                      7
total_revenue                  7
avg_ticket                     7
items_per_order_net            7
avg_days_between_orders    61971
std_days_between_orders    78984
is_one_time_buyer              7
dtype: int64


Just 7 customers returned everything they bought

In [58]:
# 1. Fill customers who returned everything
cols_to_fill = ["frequency", "total_revenue", "avg_ticket", 
                "items_per_order_net","is_one_time_buyer"]
df_train_final[cols_to_fill] = df_train_final[cols_to_fill].fillna(0)
df_valid_final[cols_to_fill] = df_valid_final[cols_to_fill].fillna(0)

# Fill one-time buyers (no days between orders)
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    median_val = df_train_final[col].median()
    df_train_final[col] = df_train_final[col].fillna(median_val)
    df_valid_final[col] = df_valid_final[col].fillna(median_val)

# Confirm no NaNs remain
print("Remaining NaNs:", df_train_final.isna().sum().sum())

Remaining NaNs: 0


## Model preparation

In [59]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

#Define feature columns 
feature_cols = [col for col in df_train_final.columns 
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
y_train = df_train_final["revenue_2018_2019"]

X_valid = df_valid_final[feature_cols]
y_valid = df_valid_final["revenue_2018_2019"]

print("Features:", feature_cols)
print("X_train shape:", X_train.shape)

Features: ['recency_days', 'frequency', 'total_revenue', 'avg_ticket', 'irems_per_order_gross', 'items_per_order_net', 'total_returns', 'total_items', 'return_rate', 'total_discounted_items', 'discount_rate', 'avg_days_between_orders', 'std_days_between_orders', 'is_one_time_buyer', 'buys_in_jan', 'buys_in_jul', 'most_active_month', 'n_genders', 'buys_women', 'buys_men', 'buys_kids', 'n_product_categories', 'n_brands', 'pct_web_only', 'buys_outlet']
X_train shape: (93272, 25)


## Binary target

Predict if the client will come back or not

In [60]:
# Stage 1 target: will the customer return or not? (1 = yes, 0 = no)
y_train_binary = (y_train > 0).astype(int)
y_valid_binary = (y_valid > 0).astype(int)

print("Train - will return:", y_train_binary.sum(), f"({100*y_train_binary.mean():.1f}%)")
print("Train - won't return:", (y_train_binary == 0).sum(), f"({100*(1-y_train_binary.mean()):.1f}%)")

Train - will return: 34076 (36.5%)
Train - won't return: 59196 (63.5%)


In [61]:
# Stage 1: classify whether the customer will return or not
clf = RandomForestClassifier(
    n_estimators=200, #number of trees
    max_depth=8, #max depth per tree 
    min_samples_leaf=20, #minimum samples required at each lead
    random_state=123, #seed
    n_jobs=-1 #use all available CPU cores
)
clf.fit(X_train, y_train_binary)

#Predict on validation set
y_pred_binary = clf.predict(X_valid)

#Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_valid_binary, y_pred_binary))

              precision    recall  f1-score   support

           0       0.72      0.87      0.79     14737
           1       0.66      0.43      0.52      8582

    accuracy                           0.71     23319
   macro avg       0.69      0.65      0.66     23319
weighted avg       0.70      0.71      0.69     23319



In [62]:
# Add class_weight="balanced"
clf = RandomForestClassifier(
    n_estimators=200, #number of trees
    max_depth=8, #max depth per tree 
    min_samples_leaf=20, #minimum samples required at each lead
    class_weight="balanced", #penalizes mistakes on the minority class more
    random_state=123, #seed
    n_jobs=-1 #use all available CPU cores
)
clf.fit(X_train, y_train_binary)

#Predict on validation set
y_pred_binary = clf.predict(X_valid)

#Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_valid_binary, y_pred_binary))

              precision    recall  f1-score   support

           0       0.75      0.77      0.76     14737
           1       0.59      0.55      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.66      0.67     23319
weighted avg       0.69      0.69      0.69     23319



Better results on recall for 1

In [63]:
#Test predicting probabilities instead of hard classes
y_pred_proba =clf.predict_proba(X_valid)[:,1] #probability of returning

#Try a lower treshold
threshold= 0.4
y_pred_binary_adjusted = (y_pred_proba >= threshold).astype(int)
print(classification_report(y_valid_binary, y_pred_binary_adjusted))

              precision    recall  f1-score   support

           0       0.77      0.62      0.68     14737
           1       0.51      0.69      0.58      8582

    accuracy                           0.64     23319
   macro avg       0.64      0.65      0.63     23319
weighted avg       0.67      0.64      0.65     23319



Worst results, we will try now grid search with the balanced class weight

In [64]:
from sklearn.model_selection import GridSearchCV

#Define the parameter grid to search
param_grid = {
    "n_estimators": [100,200,300],
    "max_depth": [6,8,10],
    "min_samples_leaf": [10,20,30],
}

#Base classifier
clf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

#Grid search with cross validation (cv=5 means 5-fold cross validtation)
# scoring="f1" because we care about both precision and recall for class 1
grid_search = GridSearchCV(
    estimator=clf_base,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    verbose=1, #print progress
    n_jobs=-1
)

grid_search.fit(X_train, y_train_binary)
print("Best parameters:", grid_search.best_params_)
print("Best CV f1-score:", grid_search.best_score_.round(3))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 30, 'n_estimators': 200}
Best CV f1-score: 0.578


In [65]:
#Train final classifier with best parameters
clf_best = grid_search.best_estimator_

#Predict on validation set
y_pred_binary_best = clf_best.predict(X_valid)
print(classification_report(y_valid_binary, y_pred_binary_best))

              precision    recall  f1-score   support

           0       0.75      0.78      0.76     14737
           1       0.59      0.55      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.66      0.67     23319
weighted avg       0.69      0.69      0.69     23319



In [66]:
# Stage 2: only train on customers who actually returned
train_returners = df_train_final[df_train_final["revenue_2018_2019"] > 0]
valid_returners = df_valid_final[df_valid_final["revenue_2018_2019"] > 0]

X_train_reg = train_returners[feature_cols]
y_train_reg = train_returners["revenue_2018_2019"]

X_valid_reg = valid_returners[feature_cols]
y_valid_reg = valid_returners["revenue_2018_2019"]

print("Train returners:", X_train_reg.shape)
print("Valid returners:", X_valid_reg.shape)

Train returners: (34076, 25)
Valid returners: (8582, 25)


In [67]:
# Stage 2: predict revenue for customers who will return
reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

reg.fit(X_train_reg, y_train_reg)

# Evaluate on returners only first
y_pred_reg = reg.predict(X_valid_reg)

mae_returners = mean_absolute_error(y_valid_reg, y_pred_reg)
spearman_returners = spearmanr(y_valid_reg, y_pred_reg).statistic

print(f"MAE (returners only):      {mae_returners:.2f}")
print(f"Spearman (returners only): {spearman_returners:.3f}")

MAE (returners only):      117.20
Spearman (returners only): 0.349


In [68]:
# Step 1: predict who will return using the classifier
y_pred_proba = clf_best.predict_proba(X_valid)[:, 1]
y_pred_will_return = (y_pred_proba >= 0.5).astype(int)

# Step 2: for predicted returners, predict revenue using the regressor
y_final_pred = np.zeros(len(X_valid))  # start everyone at 0

returner_mask = y_pred_will_return == 1
y_final_pred[returner_mask] = reg.predict(X_valid[returner_mask])

# Evaluate full pipeline
mae_full = mean_absolute_error(y_valid, y_final_pred)
spearman_full = spearmanr(y_valid, y_final_pred).statistic

print(f"MAE (full pipeline):      {mae_full:.2f}")
print(f"Spearman (full pipeline): {spearman_full:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (full pipeline):      79.60
Spearman (full pipeline): 0.400

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [69]:
from sklearn.model_selection import GridSearchCV

param_grid_reg = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 8, 10],
    "min_samples_leaf": [10, 20, 30],
}

reg_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# scoring="neg_mean_absolute_error" because GridSearchCV maximizes the score
# so we use the negative MAE (less negative = better)
grid_search_reg = GridSearchCV(
    estimator=reg_base,
    param_grid=param_grid_reg,
    cv=5,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1
)

grid_search_reg.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_search_reg.best_params_)
print("Best CV MAE:", -grid_search_reg.best_score_.round(2))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'max_depth': 6, 'min_samples_leaf': 30, 'n_estimators': 300}
Best CV MAE: 116.97


In [70]:
# Get best regressor
reg_best = grid_search_reg.best_estimator_

# Full pipeline with optimized regressor
y_final_pred_v2 = np.zeros(len(X_valid))
y_final_pred_v2[returner_mask] = reg_best.predict(X_valid[returner_mask])

mae_v2 = mean_absolute_error(y_valid, y_final_pred_v2)
spearman_v2 = spearmanr(y_valid, y_final_pred_v2).statistic

print(f"MAE (full pipeline v2):      {mae_v2:.2f}")
print(f"Spearman (full pipeline v2): {spearman_v2:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")


MAE (full pipeline v2):      79.59
Spearman (full pipeline v2): 0.400

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [71]:
from xgboost import XGBClassifier, XGBRegressor

# --- Stage 1: XGBoost Classifier ---
# scale_pos_weight handles class imbalance (equivalent to class_weight="balanced")
# it's calculated as: number of negatives / number of positives
scale = (y_train_binary == 0).sum() / (y_train_binary == 1).sum()
print(f"scale_pos_weight: {scale:.2f}")

clf_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,     # how much each tree contributes — lower = more conservative
    subsample=0.8,          # fraction of samples used per tree — reduces overfitting
    colsample_bytree=0.8,   # fraction of features used per tree — reduces overfitting
    scale_pos_weight=scale, # handles class imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

clf_xgb.fit(X_train, y_train_binary)
y_pred_binary_xgb = clf_xgb.predict(X_valid)
print(classification_report(y_valid_binary, y_pred_binary_xgb))

scale_pos_weight: 1.74
              precision    recall  f1-score   support

           0       0.75      0.77      0.76     14737
           1       0.59      0.56      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.67      0.67     23319
weighted avg       0.69      0.69      0.69     23319



In [72]:
# --- Stage 2: XGBoost Regressor ---
reg_xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

reg_xgb.fit(X_train_reg, y_train_reg)

# --- Full pipeline with XGBoost ---
y_pred_proba_xgb = clf_xgb.predict_proba(X_valid)[:, 1]
returner_mask_xgb = (y_pred_proba_xgb >= 0.5)

y_final_pred_xgb = np.zeros(len(X_valid))
y_final_pred_xgb[returner_mask_xgb] = reg_xgb.predict(X_valid[returner_mask_xgb])

mae_xgb = mean_absolute_error(y_valid, y_final_pred_xgb)
spearman_xgb = spearmanr(y_valid, y_final_pred_xgb).statistic

print(f"MAE (XGBoost pipeline):      {mae_xgb:.2f}")
print(f"Spearman (XGBoost pipeline): {spearman_xgb:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (XGBoost pipeline):      80.05
Spearman (XGBoost pipeline): 0.400

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [75]:
# Log transform the target (add 1 to avoid log(0))
# We use log1p which is log(1+x) -- a common trick for revenue/count data
y_train_reg_log = np.log1p(y_train_reg)
y_valid_reg_log = np.log1p(y_valid_reg)

# Train XGBoost regressor on log-transformed target
reg_xgb_log = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

reg_xgb_log.fit(X_train_reg, y_train_reg_log)

# Predict and convert back with expm1 (inverse of log1p)
y_pred_reg_log = np.expm1(reg_xgb_log.predict(X_valid_reg))

mae_log = mean_absolute_error(y_valid_reg, y_pred_reg_log)
spearman_log = spearmanr(y_valid_reg, y_pred_reg_log).statistic

print(f"MAE (log transform, returners only): {mae_log:.2f}")
print(f"Spearman (log transform, returners only): {spearman_log:.3f}")

MAE (log transform, returners only): 111.00
Spearman (log transform, returners only): 0.347


In [76]:
y_final_pred_log = np.zeros(len(X_valid))
y_final_pred_log[returner_mask_xgb] = np.expm1(
    reg_xgb_log.predict(X_valid[returner_mask_xgb])
)

mae_log_full = mean_absolute_error(y_valid, y_final_pred_log)
spearman_log_full = spearmanr(y_valid, y_final_pred_log).statistic

print(f"MAE (log pipeline):      {mae_log_full:.2f}")
print(f"Spearman (log pipeline): {spearman_log_full:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (log pipeline):      71.16
Spearman (log pipeline): 0.399

Target to beat -> MAE: 61.15 | Spearman: 0.41
